# Fallacy LoRA on a 7B model (QLoRA) on Colab
Runtime → Change runtime type → **T4 GPU** (an L4 or A100 is faster if you have one). 

Base model: `Qwen/Qwen2.5-7B-Instruct`, loaded in 4-bit so it fits in 16 GB. Only the LoRA adapter is trained. The run stops early if the validation loss stops improving (max 4 epochs, patience 1). Expect roughly 1.5-3 hours on a T4. Keep the browser tab open.
Data now comes from the private Hugging Face dataset `sreejithb/fallacy-dataset` (version `v2`), not from a zip.

**One-time Colab setup:** left sidebar → key icon (Secrets) → Add new secret, name `HF_TOKEN`, value = a Hugging Face **read** token, and switch on *Notebook access*. Upload `fallacy_code.zip` (code only, tiny) to the Files sidebar.


In [ ]:
!nvidia-smi -L
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate tqdm bitsandbytes

In [ ]:
# Needs fallacy_code.zip in /content (upload it first) and a Colab secret named HF_TOKEN with notebook access on.
!rm -rf /content/work && mkdir -p /content/work && unzip -oq /content/fallacy_code.zip -d /content/work
%cd /content/work
!pip -q install huggingface_hub
!python fetch_data.py --rev v2
!ls . data data/generated

## Baseline: the 7B model with no fine-tuning (optional, about 5-10 min)

In [ ]:
!python evaluate.py --base Qwen/Qwen2.5-7B-Instruct --four-bit --out baseline7b_preds.jsonl 2>&1 | tee baseline7b.log

## Run G: 7B, rank 16, early stopping

In [ ]:
!python train.py --base Qwen/Qwen2.5-7B-Instruct --four-bit --out out/lora_G --epochs 4 --patience 1 --rank 16 --batch 4 --lr 1e-4 2>&1 | tee train_G.log

In [ ]:
!python evaluate.py --base Qwen/Qwen2.5-7B-Instruct --four-bit --adapter out/lora_G --out lora_G_preds.jsonl

In [ ]:
!zip -qr lora_result_v5.zip out data/lora_G_preds.jsonl data/baseline7b_preds.jsonl train_G.log baseline7b.log
from google.colab import files
files.download('/content/work/lora_result_v5.zip')